In [1]:
import os
os.chdir("C:/journal_project")
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
def corrected_resampled_ttest(scores_a, scores_b, n_train, n_test):
    """
    Nadeau & Bengio (2003) corrected paired t-test for repeated k-fold CV.
    Adjusts for the fact that fold scores share overlapping training data,
    which inflates apparent significance under a naive paired test.
    """
    diffs = np.array(scores_a) - np.array(scores_b)
    n = len(diffs)
    mean_diff = np.mean(diffs)
    var_diff = np.var(diffs, ddof=1)

    correction = (1 / n) + (n_test / n_train)
    corrected_var = var_diff * correction
    if corrected_var <= 0:
        return np.nan, np.nan

    t_stat = mean_diff / np.sqrt(corrected_var)
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 1))
    return t_stat, p_value

In [3]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")

# StratifiedGroupKFold with 5 splits: ~80% train, ~20% test per fold
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=["hr","hrv","eda","wrist_temp","co2_noisy","lux_noisy","posture_cm"])
n_total = len(df)
n_test = n_total // 5
n_train = n_total - n_test
print(f"n_train={n_train}, n_test={n_test}")

best_model = "Proposed_Stacking_Ensemble"  # update after rerunning M10 fix if the best model changes

results = []
for col in scores_df.columns:
    if col == best_model:
        continue
    t_stat, p_val = corrected_resampled_ttest(scores_df[best_model], scores_df[col], n_train, n_test)
    print(f"{best_model} vs {col}: corrected t={t_stat:.3f}, p={p_val:.4f}")
    results.append({"comparison": f"{best_model}_vs_{col}", "t_stat": t_stat, "p_value": p_val})

pd.DataFrame(results).to_csv("outputs/corrected_statistical_test_results.csv", index=False)

n_train=1712, n_test=427
Proposed_Stacking_Ensemble vs LogisticRegression: corrected t=0.601, p=0.5506
Proposed_Stacking_Ensemble vs SVM: corrected t=2.149, p=0.0366
Proposed_Stacking_Ensemble vs RandomForest: corrected t=0.744, p=0.4602
Proposed_Stacking_Ensemble vs GradientBoosting: corrected t=0.826, p=0.4128
Proposed_Stacking_Ensemble vs MLP: corrected t=1.517, p=0.1356
Proposed_Stacking_Ensemble vs RuleEngine: corrected t=1.708, p=0.0940
Proposed_Stacking_Ensemble vs XGBoost: corrected t=0.378, p=0.7074
